# Bagging vs boosting: which base model suits which
Stumps (high bias, low variance) and fully grown trees (low bias, high variance), each used alone, bagged and
boosted, on the noisy concentric circles of the AdaBoost hyperparameters Note. 10-fold cross-validation.

In [1]:
from sklearn.datasets import make_circles
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = make_circles(n_samples=500, factor=0.1, noise=0.35, random_state=42)

In [2]:
bases = {
    "stump (max_depth=1)": lambda: DecisionTreeClassifier(max_depth=1, random_state=42),
    "fully grown tree": lambda: DecisionTreeClassifier(random_state=42),
}
for name, base in bases.items():
    models = {
        "alone": base(),
        "bagging, 100": BaggingClassifier(base(), n_estimators=100, random_state=42),
        "AdaBoost, 100": AdaBoostClassifier(base(), n_estimators=100, random_state=42),
    }
    for how, model in models.items():
        cv = cross_val_score(model, X, y, cv=10).mean()
        train = model.fit(X, y).score(X, y)
        print(f"{name:20s} {how:14s} train {train:.3f}  CV {cv:.3f}")

stump (max_depth=1)  alone          train 0.608  CV 0.566


stump (max_depth=1)  bagging, 100   train 0.848  CV 0.738


stump (max_depth=1)  AdaBoost, 100  train 0.852  CV 0.814
fully grown tree     alone          train 1.000  CV 0.746


fully grown tree     bagging, 100   train 1.000  CV 0.808
fully grown tree     AdaBoost, 100  train 1.000  CV 0.750


In [3]:
# AdaBoost with fully grown trees stops after the first tree: it already fits the training data perfectly
ada_full = AdaBoostClassifier(DecisionTreeClassifier(random_state=42), n_estimators=100,
                              random_state=42).fit(X, y)
print("trees actually trained:", len(ada_full.estimators_))

trees actually trained: 1


## Accuracy against the number of base models
Training and test accuracy after 1 and after 100 models, averaged over 20 fresh draws of the circles (350 training, 150 test points). The same function draws `images/error_curves.png`.

In [4]:
import sys
sys.path.insert(0, "images")
from error_curves import curves   # bagging: fully grown trees; boosting: AdaBoost with stumps

for (method, split), acc in curves().items():
    print(f"{method:9s} {split:5s}  1 model: {acc[0]:.2f}   100 models: {acc[-1]:.2f}")

bagging   train  1 model: 0.92   100 models: 1.00
bagging   test   1 model: 0.78   100 models: 0.82
boosting  train  1 model: 0.63   100 models: 0.88
boosting  test   1 model: 0.60   100 models: 0.84
